# ARIA - LITE

ARIA Lite is a lightweight GraphRAG-based biomedical research assistant focused on breast cancer AI literature.

The project combines two retrieval paradigms:

1. Semantic Retrieval
   Dense vector embeddings are used to retrieve papers semantically related to a user query.

2. Graph-Based Retrieval
   Biomedical entities such as genes and drugs are extracted from papers and represented as relationships in a lightweight knowledge graph.

By combining these two approaches, the system aims to provide more grounded and explainable retrieval compared to traditional vector-only RAG systems.

The project is intentionally scoped for rapid iteration and learning:
- ~300-500 PubMed papers
- Abstract-only corpus
- Lightweight graph construction
- Citation-grounded responses

Core technologies:
- PubMed / Entrez API
- SciSpacy
- Sentence Transformers
- FAISS
- Python + Google Colab

End Goal:
Build a small but functional biomedical GraphRAG system capable of retrieving relevant breast cancer AI papers and generating grounded answers with PMID citations.

# 2B_dataset_creator.ipynb

PURPOSE: Convert clean paper dataset into a structured
section-aware dataset for:

1. Chunk-level entity extraction
2. Chunk-aware GraphRAG
3. Better retrieval grounding
4. Section-aware retrieval

INPUT: clean_papers.json

OUTPUT: structured_papers.json

In [1]:
# ============================================================
# SECTION 1 — Imports
# ============================================================

from google.colab import drive
import os

import json
import re
from collections import defaultdict

In [2]:
# ============================================================
# SECTION 2 — Mount Google Drive
# ============================================================

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ============================================================
# SECTION 3 — Project Paths and data loading
# ============================================================

PROJECT_ROOT = "/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2"

INPUT_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "clean_papers.json"
)

OUTPUT_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "structured_papers.json"
)

print("Project root:", PROJECT_ROOT)
print("Input path:", INPUT_PATH)
print("Output path:", OUTPUT_PATH)

with open(INPUT_PATH, "r") as f:
    papers = json.load(f)

print(f"Loaded {len(papers)} papers")

Project root: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2
Input path: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/data/processed/clean_papers.json
Output path: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/data/processed/structured_papers.json
Loaded 475 papers


In [4]:
# ============================================================
# SECTION 4 — Utility Functions
# ============================================================

def normalize_whitespace(text):

    text = re.sub(r"\s+", " ", text)

    return text.strip()


def clean_section_text(text):

    if not text:
        return ""

    # remove HTML remnants
    text = re.sub(r"<.*?>", "", text)

    # normalize spaces
    text = normalize_whitespace(text)

    return text


def build_section_id(pmid, section_name):

    return f"{pmid}_{section_name}"


def build_chunk_id(pmid, idx):

    return f"{pmid}_chunk_{idx}"

In [5]:
# ============================================================
# SECTION 5 — Build Structured Dataset
# ============================================================

structured_papers = []

for paper in papers:

    pmid = paper.get("pmid")

    title = paper.get("title", "")

    abstract_raw = paper.get("abstract_raw", {})

    sections = []

    # --------------------------------------------------------
    # CASE 1 — Structured abstract
    # --------------------------------------------------------

    if isinstance(abstract_raw, dict):

        for idx, (section_name, section_text) in enumerate(
            abstract_raw.items()
        ):

            section_text = clean_section_text(section_text)

            if len(section_text) == 0:
                continue

            sections.append({

                "section_id": build_section_id(
                    pmid,
                    section_name
                ),

                "chunk_id": build_chunk_id(
                    pmid,
                    idx
                ),

                "section_type": section_name,

                "text": section_text
            })

    # --------------------------------------------------------
    # CASE 2 — Unstructured abstract
    # --------------------------------------------------------

    elif isinstance(abstract_raw, str):

        abstract_text = clean_section_text(
            abstract_raw
        )

        if len(abstract_text) > 0:

            sections.append({

                "section_id": build_section_id(
                    pmid,
                    "ABSTRACT"
                ),

                "chunk_id": build_chunk_id(
                    pmid,
                    0
                ),

                "section_type": "ABSTRACT",

                "text": abstract_text
            })

    # --------------------------------------------------------
    # Final paper structure
    # --------------------------------------------------------

    structured_papers.append({

        "pmid": pmid,

        "title": title,

        "year": paper.get("year"),

        "journal": paper.get("journal"),

        "authors": paper.get("authors", []),

        # keep original flattened text
        "text": paper.get("text", ""),

        # NEW structured sections
        "sections": sections
    })

print(f"\nStructured papers created: {len(structured_papers)}")


Structured papers created: 475


In [6]:
# ============================================================
# SECTION 6 — Dataset Statistics
# ============================================================

total_sections = 0

section_counter = defaultdict(int)

for paper in structured_papers:

    total_sections += len(
        paper["sections"]
    )

    for section in paper["sections"]:

        section_counter[
            section["section_type"]
        ] += 1

print("\n===================================")
print("DATASET STATISTICS")
print("===================================")

print("Total papers:", len(structured_papers))
print("Total sections:", total_sections)

print("\nSection distribution:")

for section_name, count in sorted(
    section_counter.items()
):

    print(f"{section_name}: {count}")


DATASET STATISTICS
Total papers: 475
Total sections: 1135

Section distribution:
ADVANCES IN KNOWLEDGE: 1
AIM: 2
AIMS: 2
APPROACH: 3
AVAILABILITY: 1
BACKGROUND: 101
BACKGROUND AND AIMS: 1
BACKGROUND AND OBJECTIVE: 2
BACKGROUND AND PURPOSE: 2
BACKGROUND/AIM: 3
BACKGROUND/OBJECTIVES: 1
BACKGROUNDS: 1
CASE PRESENTATION: 1
CASE REPORT: 1
CLINICAL TRIAL INFORMATION: 1
CLINICAL TRIAL NUMBER: 1
CLINICAL TRIAL REGISTRATION NUMBER: 1
CLINICALTRIALS GOV IDENTIFIER: 1
CONCLUSION: 88
CONCLUSION AND CLINICAL INTERPRETATION: 1
CONCLUSIONS: 85
CONCLUSIONS AND RELEVANCE: 4
CONCLUSIONS.—: 1
CONTEXT.—: 1
DATA FORMAT AND USAGE NOTES: 1
DESIGN: 1
DESIGN, SETTING, AND PARTICIPANTS: 4
DESIGN.—: 1
DEVELOPMENT AND VALIDATION METHODS: 1
DIAGNOSTIC PREDICTIVE AND PROGNOSTIC APPLICATION: 1
DISCUSSION: 9
ETHICS AND DISSEMINATION: 1
EXPERIMENTAL APPROACH: 1
EXPERIMENTAL DESIGN: 1
EXPOSURE: 1
EXPOSURES: 2
FINDINGS: 8
FUNDING: 6
GRAPHICAL ABSTRACT.: 1
IMPLICATIONS: 1
IMPLICATIONS FOR CANCER SURVIVORS: 2
IMPORTANCE:

In [7]:
# ============================================================
# SECTION 7 — Example Inspection
# ============================================================

sample_paper = structured_papers[0]

print("\n===================================")
print("SAMPLE PAPER")
print("===================================")

print("PMID:", sample_paper["pmid"])

print("\nTITLE:")
print(sample_paper["title"])

print("\nSECTIONS:")

for section in sample_paper["sections"]:

    print("\n-----------------------------")

    print("Section ID:",
          section["section_id"])

    print("Chunk ID:",
          section["chunk_id"])

    print("Section Type:",
          section["section_type"])

    print("Text Preview:")

    print(section["text"][:300])


SAMPLE PAPER
PMID: 42104431

TITLE:
Leveraging immune and clinicopathological profiles with machine learning to predict axillary lymph node metastasis in breast cancer patients.

SECTIONS:

-----------------------------
Section ID: 42104431_BACKGROUND
Chunk ID: 42104431_chunk_0
Section Type: BACKGROUND
Text Preview:
Breast cancer is the leading cause of cancer-related death in women, with mortality increasing when tumor cells spread to nearby lymph nodes, particularly the axillary lymph nodes (ALNs). Although several studies predict patients with ALN metastasis at diagnosis (pdALN+), few examine the prognostic 

-----------------------------
Section ID: 42104431_METHODS
Chunk ID: 42104431_chunk_1
Section Type: METHODS
Text Preview:
Two datasets of luminal breast cancer patients diagnosed between 1995 and 2008 were used: Dataset 1 involved 83 women (42 pdALN- and 41 pdALN+), and Dataset 2 comprised 344 women (204 pdALN- and 140 pdALN+). Three machine learning models were developed usin

In [8]:
# ============================================================
# SECTION 8 — Save Structured Dataset
# ============================================================

with open(OUTPUT_PATH, "w") as f:

    json.dump(
        structured_papers,
        f,
        indent=2
    )

print("\n===================================")
print("SAVE COMPLETE")
print("===================================")

print("Saved structured dataset to:")
print(OUTPUT_PATH)


SAVE COMPLETE
Saved structured dataset to:
/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/data/processed/structured_papers.json
